In [5]:
# testing.ipynb - single-row pipeline test

import os
import json
from datetime import datetime

try:
    import pandas as pd
except ModuleNotFoundError as e:
    raise RuntimeError(
        "pandas is required to run this notebook. Install it in the same kernel environment. "
        "For this repo: run './.venv/bin/python -m pip install pandas'"
    ) from e

from app.orchestrator.workflow import process_complaint

CSV_PATH = "complaint_data/split_file_0.csv"
DEFAULT_COMPANY_ID = os.getenv("COMPANY_ID", "mock_bank")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("CSV_PATH:", CSV_PATH)
print("DEFAULT_COMPANY_ID:", DEFAULT_COMPANY_ID)

# Load the CSV header + data
# (for simplicity we load full file once; if this is too large, you can
# switch to chunked reading and stop after the first valid row).
df = pd.read_csv(CSV_PATH)
print("Loaded row columns:")
print(list(df.columns))



OPENAI_API_KEY set: True
CSV_PATH: complaint_data/split_file_0.csv
DEFAULT_COMPANY_ID: mock_bank
Loaded row columns:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [6]:

def get_first_existing(col_candidates):
    for c in col_candidates:
        if c in df.columns:
            return c
    return None


# CFPB-style column names used by app/retrieval/ingest.py
COL_NARRATIVE = get_first_existing([
    "Consumer complaint narrative",
    "consumer_narrative",
    "narrative",
])
COL_PRODUCT = get_first_existing(["Product", "product"])
COL_SUB_PRODUCT = get_first_existing(["Sub-product", "sub_product"])
COL_COMPANY = get_first_existing(["Company", "company"])
COL_STATE = get_first_existing(["State", "state"])
COL_ZIP = get_first_existing(["ZIP code", "Zip code", "zip_code", "ZIP"])
COL_CHANNEL = get_first_existing(["Submitted via", "Channel", "channel"])
COL_RESPONSE = get_first_existing(["Company response to consumer", "Company response"])
COL_DATE_RECEIVED = get_first_existing(["Date received", "date_received"])
COL_ISSUE = get_first_existing(["Issue", "issue"])

missing = [
    name
    for name, val in [
        ("narrative", COL_NARRATIVE),
        ("issue", COL_ISSUE),
    ]
    if val is None
]

if missing:
    raise RuntimeError(
        "CSV is missing required columns for the test: " + ", ".join(missing)
    )

print("Using columns:")
print({
    "narrative": COL_NARRATIVE,
    "issue": COL_ISSUE,
    "product": COL_PRODUCT,
    "sub_product": COL_SUB_PRODUCT,
    "company": COL_COMPANY,
    "state": COL_STATE,
    "zip_code": COL_ZIP,
    "channel": COL_CHANNEL,
    "date_received": COL_DATE_RECEIVED,
    "requested_resolution": COL_RESPONSE,
})


Using columns:
{'narrative': 'Consumer complaint narrative', 'issue': 'Issue', 'product': 'Product', 'sub_product': 'Sub-product', 'company': 'Company', 'state': 'State', 'zip_code': 'ZIP code', 'channel': 'Submitted via', 'date_received': 'Date received', 'requested_resolution': 'Company response to consumer'}


In [7]:

# Pick the first row with a non-null, reasonably long narrative
valid_mask = df[COL_NARRATIVE].notna() & (df[COL_NARRATIVE].astype(str).str.len() >= 10)
if not valid_mask.any():
    raise RuntimeError("No rows in the CSV have a narrative with at least 10 characters.")

row = df.loc[valid_mask].iloc[0]

# Map submitted-via/channel strings into CaseCreate.channel enum values
channel_raw = str(row[COL_CHANNEL]).strip() if COL_CHANNEL else "web"
channel_raw_lower = channel_raw.lower()

channel_map = {
    "web": "web",
    "online": "web",
    "phone": "phone",
    "email": "email",
    "fax": "fax",
    "postal": "postal",
    "mail": "postal",
    "referral": "referral",
}
channel = channel_map.get(channel_raw_lower, "web")

submitted_at = None
if COL_DATE_RECEIVED:
    val = row[COL_DATE_RECEIVED]
    if pd.notna(val):
        try:
            submitted_at = pd.to_datetime(val, errors="coerce").to_pydatetime()
        except Exception:
            submitted_at = None

narrative_val = row[COL_NARRATIVE]
if pd.isna(narrative_val) or len(str(narrative_val).strip()) < 10:
    raise RuntimeError("Selected row has an invalid narrative; this should not happen if valid_mask is correct.")

payload = {
    "company_id": DEFAULT_COMPANY_ID,
    "consumer_narrative": str(narrative_val),
    "product": (str(row[COL_PRODUCT]).strip() if COL_PRODUCT else None) or None,
    "sub_product": (str(row[COL_SUB_PRODUCT]).strip() if COL_SUB_PRODUCT else None) or None,
    "company": (str(row[COL_COMPANY]).strip() if COL_COMPANY else None) or None,
    "state": (str(row[COL_STATE]).strip() if COL_STATE else None) or None,
    "zip_code": (str(row[COL_ZIP]).strip() if COL_ZIP else None) or None,
    "channel": channel,
    "submitted_at": submitted_at.isoformat() if submitted_at else None,
    # External labels into the new company-aware schema boundary
    "external_product_category": (str(row[COL_PRODUCT]).strip() if COL_PRODUCT else None) or None,
    "external_issue_type": (str(row[COL_ISSUE]).strip() if COL_ISSUE else None) or None,
    "requested_resolution": (str(row[COL_RESPONSE]).strip() if COL_RESPONSE else None) or None,
}

print("\nConstructed CaseCreate payload (trimmed):")
print({k: (str(v)[:120] + '...' if isinstance(v, str) and len(v) > 120 else v) for k, v in payload.items()})

if not OPENAI_API_KEY:
    print("\nOPENAI_API_KEY is not set; skipping the LLM-powered pipeline run.")
    print("You can set OPENAI_API_KEY in your environment, then rerun the notebook.")
else:
    # Note: python-dotenv loads variables when the app imports DB/session modules.
    # If this still prints False above, ensure your kernel CWD is the repo root
    # and that `.env` contains OPENAI_API_KEY.
    final_state = process_complaint(payload)
    case = final_state["case"]

    print("\nPipeline completed.")
    print("Routed to:", case.get("routed_to"))

    print("\nClassification:")
    print(json.dumps(case.get("classification"), indent=2, default=str))

    print("\nRisk assessment:")
    print(json.dumps(case.get("risk_assessment"), indent=2, default=str))

    print("\nRoot-cause hypothesis:")
    print(json.dumps(case.get("root_cause_hypothesis"), indent=2, default=str))

    print("\nProposed resolution:")
    print(json.dumps(case.get("proposed_resolution"), indent=2, default=str))

    print("\nCompliance flags:")
    print(json.dumps(case.get("compliance_flags"), indent=2, default=str))

    print("\nEvidence trace (first 3 items if present):")
    et = case.get("evidence_trace")
    if isinstance(et, dict) and isinstance(et.get("items"), list):
        print(json.dumps({"items": et["items"][:3]}, indent=2, default=str))
    else:
        print(json.dumps(et, indent=2, default=str))




Constructed CaseCreate payload (trimmed):
{'company_id': 'mock_bank', 'consumer_narrative': 'These are not my accounts.', 'product': 'Credit reporting, credit repair services, or other personal consumer reports', 'sub_product': 'Credit reporting', 'company': 'Experian Information Solutions Inc.', 'state': 'NV', 'zip_code': '89030', 'channel': 'web', 'submitted_at': '2020-05-08T00:00:00', 'external_product_category': 'Credit reporting, credit repair services, or other personal consumer reports', 'external_issue_type': 'Incorrect information on your report', 'requested_resolution': 'Closed with explanation'}


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [7]:
import os, pathlib

print("Kernel CWD:", pathlib.Path().resolve())
print("Has OPENAI_API_KEY:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_API_KEY length:", len(os.getenv("OPENAI_API_KEY","")))
print("Repo .env exists here:", (pathlib.Path(".env").resolve()).exists())

Kernel CWD: /Users/nandanprince/Desktop/AI Agent for Complaint Classification
Has OPENAI_API_KEY: False
OPENAI_API_KEY length: 0
Repo .env exists here: True


### Test run notes

- This notebook loads **one row** from `complaint_data/split_file_0.csv`.
- It constructs a `CaseCreate` payload and calls `process_complaint(payload)`.
- If `OPENAI_API_KEY` is **not** set, it prints the payload and skips the LLM pipeline so you can verify CSV parsing/field mapping first.
- After setting `OPENAI_API_KEY`, rerun the last cell to execute the full multi-agent pipeline.
